In [ ]:
import sys

sys.path.append("..")

import os
from datetime import datetime
from glob import glob

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

from nnspike.data import (
    BrightnessAdjustDataset,
    adjust_brightness_contrast,
    revert_brightness_contrast,
)
from nnspike.models import BetaPredictorLite
from nnspike.utils import (
    load_checkpoint,
    save_checkpoint,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_label = "beta"
date_label = datetime.today().strftime("%m%d")

print(f"Train the model on '{device}'")
print(f"Model Label: {model_label}, Date Label: {date_label}")

## Calculate Raw Data Brightness

In [ ]:
label_paths = glob("../storage/labels/*.csv")
label_paths = [path for path in label_paths if os.path.basename(path)[:8] > "20250801"]

df = pd.DataFrame()
for label_path in label_paths:
    label_df = pd.read_csv(label_path)
    df = pd.concat([df, label_df])

df = df[df["use"] == True]

df = df.reset_index(drop=True)  # Reset index for future data balancing
print(f"Total number of training records: {len(df)}")

In [ ]:
brightness = []
indices_to_drop = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Processing DataFrame"):
    image_path = row["image_path"]
    image = cv2.imread(image_path)
    # Convert to HSV color space
    hsv_image = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # Extract the Value (V) channel
    v_channel = hsv_image[:, :, 2]

    # Calculate the average brightness
    average_brightness = np.mean(v_channel)

    brightness.append(average_brightness)

df["brightness"] = brightness
df.to_csv("../storage/beta/brightness.csv", index=False)

## Visualize Brightness Distribution

In [ ]:
df = pd.read_csv("../storage/gamma/brightness.csv", dtype={"brightness": float})
plt.figure(figsize=(10, 5))
counts, bins, patches = plt.hist(df["brightness"], bins=20, edgecolor="black")

counts_threshold = 1000
counts = np.asarray(counts)

mask = counts > counts_threshold

if mask.any():
    bin_width = bins[1] - bins[0]
    lower_bound = bins[:-1][mask].min()
    upper_bound = bins[:-1][mask].max() + bin_width
    filtered_df = df[
        (df["brightness"] >= lower_bound) & (df["brightness"] < upper_bound)
    ]
else:
    print(f"No bins exceed threshold {counts_threshold}; using full dataset.")
    filtered_df = df

print(f"Images trainable counts: {len(filtered_df)} (Threshold: {counts_threshold})")
df = filtered_df

plt.title("Distribution of ColumnA")
plt.xlabel("Value")
plt.ylabel("Frequency")
plt.show()

## Data Augmentation by Adjusting Beta

In [ ]:
import random

import albumentations as A  # noqa: N812
import cv2

# from nnspike.data import random_shift_scale_rotate

# Initialize lists to store image_path and beta values
adjusted_paths = []
betas = []
beta_df = pd.DataFrame()

for i, row in tqdm(df.iterrows(), total=len(df), desc="Processing DataFrame"):
    image_path = row["image_path"]
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Convert to RGB

    # image, _ = random_shift_scale_rotate(image)
    beta = random.random()  # Generates a random float N such that 0.0 <= N < 1.0

    adjusted_image = adjust_brightness_contrast(image=image, alpha=1, beta=beta * 100)

    adjusted_path = f"../storage/beta/adjusted/{i}.png"

    cv2.imwrite(adjusted_path, adjusted_image)

    # Save image_path and beta value
    adjusted_paths.append(adjusted_path)
    betas.append(beta)

# After the loop, add the data back to the DataFrame or save to a new DataFrame
beta_df["adjusted_paths"] = adjusted_paths
beta_df["beta"] = betas

# Optional: Save to CSV
beta_df.to_csv("../storage/beta/label.csv", index=False)

## Loading Training Data

In [ ]:
df = pd.read_csv("../storage/beta/label.csv")

adjusted_paths = df["adjusted_paths"].to_list()
betas = df["beta"].to_list()

X_all = adjusted_paths
y_all = betas

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, test_size=0.3, random_state=42
)

train_set = BrightnessAdjustDataset(inputs=X_train, outputs=y_train)
val_set = BrightnessAdjustDataset(inputs=X_val, outputs=y_val)

# Use `inputs, outputs = next(iter(train_loader))` for debugging
train_loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_set, batch_size=64, shuffle=True)

In [ ]:
criterion = nn.SmoothL1Loss(beta=0.05)  # or `criterion = nn.MSELoss()`
model = BetaPredictorLite()
model.to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-2)

# Learning rate scheduler - reduce on plateau
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=10
)

## Training Loop

In [ ]:
# Initialize TensorBoard writer
writer = SummaryWriter()

# Training loop
num_epochs = 10
best_val_loss = float("inf")
best_model_state = None
best_epoch = 0

# Training loop
for epoch in range(num_epochs):
    # Training
    model.train()
    train_loss = 0.0
    train_mode_loss = 0.0
    train_control_loss = 0.0
    for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}"):
        inputs = inputs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Calculate average losses
    avg_train_loss = train_loss / len(train_loader)

    # Validation
    model.eval()
    val_loss = 0.0
    for inputs, labels in val_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        with torch.no_grad():
            outputs = model(inputs)
        loss = criterion(outputs, labels)
        val_loss += loss.item()

    # Calculate average validation losses
    avg_val_loss = val_loss / len(val_loader)

    # Update learning rate scheduler
    scheduler.step(avg_val_loss)

    # Check if this is the best model so far
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_epoch = epoch + 1
        save_checkpoint(
            model,
            optimizer,
            epoch=epoch,
            loss=best_val_loss,
            save_path=f"../storage/checkpoints/{model_label}_{date_label}.pth",
        )
        print(f"New best model found at epoch {best_epoch}!")

    # Log to TensorBoard
    writer.add_scalar("Loss/train_total", avg_train_loss, epoch)
    writer.add_scalar("Loss/val_total", avg_val_loss, epoch)
    writer.add_scalar("Loss/best_val", best_val_loss, epoch)

    print(f"Epoch {epoch + 1}/{num_epochs}:")
    print(f"  Train - Total: {avg_train_loss:.5f}")
    print(f"  Val   - Total: {avg_val_loss:.5f}")
    print(f"  Learning Rate: {optimizer.param_groups[0]['lr']:.6f}")

# Load the best model state
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(
        f"\nLoaded best model from epoch {best_epoch} with validation loss: {best_val_loss:.5f}"
    )
else:
    print("\nNo best model found, keeping final model state")

# Close TensorBoard writer
writer.close()

print("\nTraining completed! Feature maps have been logged to TensorBoard.")
print("To view the log, run: tensorboard --logdir=runs")

In [ ]:
model = BetaPredictorLite()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

epoch, loss = load_checkpoint(
    model=model,
    checkpoint_path=f"../storage/checkpoints/{model_label}_{date_label}.pth",
    optimizer=optimizer,
    device=str(device),
)

## Pytorch Model Inference

In [ ]:
image = cv2.imread("./sample.png")
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

image = adjust_brightness_contrast(image, 1, 20)

# Normalize to [0, 1] range
image = image.astype(np.float32) / 255.0

# Convert to PyTorch tensor: (H, W, C) -> (C, H, W)
tensor_image = torch.from_numpy(image).permute(2, 0, 1)
tensor_image = tensor_image.to(torch.float32)
tensor_image = tensor_image.to(device)

model.eval()
with torch.no_grad():
    outputs = model(tensor_image.unsqueeze(0))

beta = outputs[0][0].item() * 100
print(f"Predicted beta is: {beta}")

## Export to ONNX & Inference

In [ ]:
onnx_path = f"../storage/models/{model_label}_{date_label}.onnx"
# Create dummy inputs - adjust dimensions as needed
dummy_image = torch.randn(1, 3, 480, 640).to(device)

# Export to ONNX
torch.onnx.export(
    model,
    (dummy_image),
    onnx_path,
    export_params=True,
    input_names=["image"],
    output_names=["beta"],
)

In [ ]:
import onnxruntime as ort

# Load the ONNX model
onnx_path = f"../storage/models/{model_label}_{date_label}.onnx"
session = ort.InferenceSession(onnx_path)

image = cv2.imread(
    "./sample.png", cv2.IMREAD_COLOR
)  # or use dummy image `dummy_image = np.random.randn(1, 3, 480, 640).astype(np.float32)`
image = adjust_brightness_contrast(image, 1, 90)

# Convert BGR to RGB (OpenCV uses BGR by default)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Convert to float and normalize to [0, 1] range
image = image.astype(np.float32) / 255.0  # Normalize
image = np.transpose(image, (2, 0, 1))  # HWC to CHW
image = np.expand_dims(image, axis=0)  # Add batch dimension

# Prepare inputs dictionary
inputs = {"image": image}

# Run inference
outputs = session.run(["beta"], inputs)

# Get results
adjustment = outputs[0][0] * 100

# beta output
print(f"Control output: {adjustment}")

## Visualize Results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

image = cv2.imread("./sample.png")
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

image = adjust_brightness_contrast(image, 1, 70)

adjusted_image = image.copy()
adjusted_image = adjusted_image.astype(np.float32) / 255.0  # Normalize
adjusted_image = np.transpose(adjusted_image, (2, 0, 1))  # HWC to CHW
adjusted_image = np.expand_dims(adjusted_image, axis=0)  # Add batch dimension

# Prepare inputs dictionary
inputs = {"image": adjusted_image}

# Run inference
outputs = session.run(["beta"], inputs)
beta = outputs[0][0]

reverted_image = revert_brightness_contrast(image, 1, beta * 100)

fig, axs = plt.subplots(1, 2, figsize=(10, 4))  # 1 row, 3 columns

# Display each image in its respective subplot
axs[0].imshow(image)
axs[0].set_title("Image 1 (Original)")
axs[0].axis("off")  # Turn off axes for cleaner image display

axs[1].imshow(reverted_image)  # Specify colormap for grayscale
axs[1].set_title("Image 2 (Adjusted)")
axs[1].axis("off")

# Adjust layout to prevent overlap
plt.tight_layout()

# Show the plot
plt.show()